# Ground Settlement Calculation During Dewatering (`bronbemaling`)

**Worked Example Analysis — Dewatering of a Construction Pit in Flanders, Belgium**  
*Author: Kherim Willems* | *Package: bronbemaling v0.1.0*

---

## Executive Summary
This notebook demonstrates the complete end-to-end workflow for analyzing ground settlement (*zetting*) and neighboring building damage risk caused by construction pit dewatering (*bronbemaling*).

### Problem Scenario:
- **Location**: Flemish lowland site (typical soil profile: fill, sand, compressible clay, deep sand).
- **Excavation Pit**: $10\text{ m} \times 8\text{ m}$ rectangular excavation, $3.0\text{ m}$ depth below surface ($2.0\text{ mTAW}$).
- **Groundwater Table**: Original GWL at $4.0\text{ mTAW}$ ($1.0\text{ m}$ below ground surface at $5.0\text{ mTAW}$).
- **Dewatering System**: 6 peripheral extraction wells pumping down to target level $1.5\text{ mTAW}$ ($2.5\text{ m}$ total drawdown in pit).
- **Neighboring Building**: Masonry residential structure located $12.0\text{ m}$ from pit center.

In [ ]:
from functools import partial
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from bronbemaling import (
    SoilProfile, SoilLayer, ConstructionPit, Well, DewateringConfig, Building,
    AquiferType, BuildingType,
    compute_drawdown_at_points, compute_drawdown_grid,
    compute_initial_stress_profile, compute_stress_increase_from_drawdown,
    compute_total_settlement, compute_settlement_vs_time,
    assess_building_damage, classify_damage,
    create_grid, solve_steady_state, extract_drawdown_at_points,
    plot_cross_section, plot_plan_view, plot_settlement_trough,
    plot_time_settlement, plot_effective_stress_profile,
    plot_3d_drawdown, plot_damage_summary,
)

print("bronbemaling package successfully loaded.")

## §1 Input Parameters (Invoergegevens)

In [ ]:
# === Soil Profile (Grondopbouw) ===
profile = SoilProfile(
    surface_level_mtaw=5.0,  # mTAW
    gwl_mtaw=4.0,            # mTAW (1m below surface)
    layers=[
        SoilLayer("Aanvulling (Fill)", thickness=0.5, gamma=17.0, gamma_sat=19.0,
                  k_h=1e-5, e0=0.6, Cc=0.05, Cr=0.01, Eoed=15000, Cv=1e-4, OCR=3.0),
        SoilLayer("Zand (Sand)", thickness=2.0, gamma=17.5, gamma_sat=20.0,
                  k_h=1e-4, e0=0.5, Cc=0.02, Cr=0.005, Eoed=30000, Cv=1e-2, OCR=1.5),
        SoilLayer("Klei (Clay)", thickness=3.0, gamma=16.0, gamma_sat=18.5,
                  k_h=1e-9, e0=1.0, Cc=0.30, Cr=0.06, Eoed=3000, Cv=1e-7, OCR=1.5),
        SoilLayer("Zand (Sand, deep)", thickness=4.5, gamma=18.0, gamma_sat=20.5,
                  k_h=5e-4, e0=0.45, Cc=0.01, Cr=0.003, Eoed=40000, Cv=1e-2, OCR=1.0),
    ]
)

# === Construction Pit (Bouwput) ===
pit = ConstructionPit(length=10.0, width=8.0, depth=3.0, 
                      center_x=0.0, center_y=0.0, bottom_mtaw=2.0)

# === Dewatering Wells (Bronnen) ===
wells = [
    Well(x=-5.5, y=-4.5, Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
    Well(x=0.0,  y=-4.5, Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
    Well(x=5.5,  y=-4.5, Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
    Well(x=-5.5, y=4.5,  Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
    Well(x=0.0,  y=4.5,  Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
    Well(x=5.5,  y=4.5,  Q=0.0005, screen_top_mtaw=3.0, screen_bottom_mtaw=0.0),
]

# === Dewatering Configuration (Bemaling) ===
dewatering = DewateringConfig(
    wells=wells,
    target_drawdown_mtaw=1.5,   # mTAW (pump down to 1.5 mTAW)
    original_gwl_mtaw=4.0,      # mTAW
    pumping_duration_days=90,
    aquifer_type=AquiferType.UNCONFINED,
)

# === Neighboring Building (Naburig Gebouw) ===
building = Building(x=12.0, y=0.0, length=10.0, width=6.0,
                    foundation_depth=0.6, building_type=BuildingType.MASONRY)

print(f"Soil Profile Total Depth: {profile.total_depth:.1f} m")
print(f"GWL Depth Below Surface: {profile.gwl_depth:.1f} m")
print(f"Target Dewatering Drawdown: {dewatering.target_drawdown:.1f} m")

## §2 Drawdown Calculation (Verlagingsberekening)

In [ ]:
b_pts = building.evaluation_points()
drawdowns_b = compute_drawdown_at_points(b_pts, dewatering, profile)
print(f"Drawdown at Building Center: {drawdowns_b[0]:.3f} m")
for i, d in enumerate(drawdowns_b[1:], 1):
    print(f"Drawdown at Corner {i}: {d:.3f} m")

X_grid, Y_grid, S_grid = compute_drawdown_grid(
    x_range=(-50, 50), y_range=(-50, 50), nx=50, ny=50,
    config=dewatering, profile=profile
)

In [ ]:
fig_cs = plot_cross_section(profile, pit, dewatering, building, drawdown_at_building=drawdowns_b[0])
plt.show()

In [ ]:
drawdown_func = partial(compute_drawdown_at_points, config=dewatering, profile=profile)
assessment = assess_building_damage(building, profile, dewatering, drawdown_func)
fig_plan = plot_plan_view(pit, dewatering, building, X_grid, Y_grid, S_grid, assessment)
plt.show()

In [ ]:
fig_3d = plot_3d_drawdown(X_grid, Y_grid, S_grid, pit, building)
fig_3d.show()

## §3 Settlement Calculation (Zettingsberekening)

In [ ]:
z_pts, sigma_v0_eff, sigma_v0_tot = compute_initial_stress_profile(profile)
_, dsigma_eff = compute_stress_increase_from_drawdown(profile, drawdown=drawdowns_b[0])
total_s_m, layer_s_m = compute_total_settlement(profile, drawdown=drawdowns_b[0], method="cc_cr")

print(f"Total Settlement at Building Center: {total_s_m * 1000.0:.2f} mm\n")
print("Per-Layer Settlement Breakdown:")
for layer, s in zip(profile.layers, layer_s_m):
    print(f" - {layer.name:<20}: {s * 1000.0:.2f} mm")

In [ ]:
sigma_vf_eff = sigma_v0_eff + dsigma_eff
fig_stress = plot_effective_stress_profile(profile, z_pts, sigma_v0_eff, sigma_vf_eff)
plt.show()

In [ ]:
x_transect = np.linspace(0, 40, 100)
points_transect = [(x, 0.0) for x in x_transect]
drawdowns_transect = compute_drawdown_at_points(points_transect, dewatering, profile)
settlements_transect = [compute_total_settlement(profile, d, method="cc_cr")[0] for d in drawdowns_transect]

fig_trough = plot_settlement_trough(profile, dewatering, pit, building, x_transect, np.array(settlements_transect))
plt.show()

## §4 Time-Dependent Consolidation (Tijdsafhankelijke Consolidatie)

In [ ]:
times_days = np.linspace(0, 365, 100)
corner_names = ["center", "corner_1", "corner_2", "corner_3", "corner_4"]
settlements_corners_time = {}

for idx, pt in enumerate(b_pts):
    d_pt = drawdowns_b[idx]
    s_time = compute_settlement_vs_time(profile, drawdown=d_pt, times_days=times_days, method="cc_cr")
    settlements_corners_time[corner_names[idx]] = s_time

fig_time = plot_time_settlement(times_days, settlements_corners_time, dewatering.pumping_duration_days)
plt.show()

## §5 Damage Assessment (Schade-beoordeling)

In [ ]:
print(f"Max Settlement:             {assessment.max_settlement * 1000.0:.2f} mm")
print(f"Min Settlement:             {assessment.min_settlement * 1000.0:.2f} mm")
print(f"Differential Settlement:    {assessment.differential_settlement * 1000.0:.2f} mm")
print(f"Angular Distortion β:       1/{int(1.0/max(assessment.angular_distortion, 1e-9))}")
print(f"Deflection Ratio Δ/L:       {assessment.deflection_ratio:.6f}")
print(f"Damage Category:            {assessment.damage_category} ({assessment.damage_description})")
print(f"Expected Crack Width:       {assessment.expected_crack_width}")

fig_damage = plot_damage_summary(assessment)
plt.show()

## §6 Numerical Method — Finite Difference (Numerieke Methode)

In [ ]:
fd_grid = create_grid(x_range=(-50, 50), y_range=(-50, 50), dx=2.0)
fd_grid = solve_steady_state(fd_grid, dewatering, profile, pit)
s_fd = extract_drawdown_at_points(fd_grid, b_pts, dewatering.original_gwl_mtaw)

print("Comparison of Drawdown at Building Evaluation Points (Analytical vs Numerical FD):")
for idx, (d_an, d_num) in enumerate(zip(drawdowns_b, s_fd)):
    print(f" Point {corner_names[idx]:<10}: Analytical = {d_an:.3f} m | Numerical FD = {d_num:.3f} m")

## §7 Sensitivity Analysis (Gevoeligheidsanalyse)

In [ ]:
# Sensitivity 1: Building Distance
distances = np.linspace(5.0, 30.0, 20)
s_dist = []
for dist in distances:
    b_temp = Building(x=dist, y=0.0, length=10.0, width=6.0)
    pts_temp = b_temp.evaluation_points()
    d_temp = compute_drawdown_at_points(pts_temp, dewatering, profile)[0]
    s_temp = compute_total_settlement(profile, d_temp)[0]
    s_dist.append(s_temp * 1000.0)

plt.figure(figsize=(8, 4))
plt.plot(distances, s_dist, 'o-', color='purple')
plt.xlabel("Building Center X-Distance [m]")
plt.ylabel("Settlement [mm]")
plt.title("Sensitivity 1: Settlement vs Building Distance")
plt.grid(True, linestyle=":")
plt.show()

In [ ]:
# Sensitivity 2: Target Drawdown
target_drawdowns = np.linspace(0.5, 3.0, 15)
s_dd = []
for dd in target_drawdowns:
    cfg_temp = DewateringConfig(wells=wells, target_drawdown_mtaw=4.0 - dd, original_gwl_mtaw=4.0, pumping_duration_days=90)
    d_temp = compute_drawdown_at_points([b_pts[0]], cfg_temp, profile)[0]
    s_temp = compute_total_settlement(profile, d_temp)[0]
    s_dd.append(s_temp * 1000.0)

plt.figure(figsize=(8, 4))
plt.plot(target_drawdowns, s_dd, 's-', color='teal')
plt.xlabel("Target Drawdown in Pit [m]")
plt.ylabel("Settlement at Building [mm]")
plt.title("Sensitivity 2: Settlement vs Target Drawdown")
plt.grid(True, linestyle=":")
plt.show()

In [ ]:
# Sensitivity 3: Clay Layer Thickness
clay_thicknesses = np.linspace(1.0, 6.0, 15)
s_clay = []
for t_clay in clay_thicknesses:
    prof_temp = SoilProfile(
        surface_level_mtaw=5.0, gwl_mtaw=4.0,
        layers=[
            SoilLayer("Aanvulling", 0.5, 17.0, 19.0, 1e-5, 0.6, 0.05, 0.01, 15000, 1e-4),
            SoilLayer("Zand", 2.0, 17.5, 20.0, 1e-4, 0.5, 0.02, 0.005, 30000, 1e-2),
            SoilLayer("Klei", t_clay, 16.0, 18.5, 1e-9, 1.0, 0.30, 0.06, 3000, 1e-7),
            SoilLayer("Zand diep", 4.5, 18.0, 20.5, 5e-4, 0.45, 0.01, 0.003, 40000, 1e-2),
        ]
    )
    d_temp = compute_drawdown_at_points([b_pts[0]], dewatering, prof_temp)[0]
    s_temp = compute_total_settlement(prof_temp, d_temp)[0]
    s_clay.append(s_temp * 1000.0)

plt.figure(figsize=(8, 4))
plt.plot(clay_thicknesses, s_clay, 'd-', color='darkgreen')
plt.xlabel("Clay Layer Thickness [m]")
plt.ylabel("Settlement at Building [mm]")
plt.title("Sensitivity 3: Settlement vs Clay Layer Thickness")
plt.grid(True, linestyle=":")
plt.show()

In [ ]:
# Sensitivity 4: Clay Permeability (Cv)
cv_values = np.logspace(-8, -5, 15)
t_50_days = []
for cv in cv_values:
    prof_temp = SoilProfile(
        surface_level_mtaw=5.0, gwl_mtaw=4.0,
        layers=[
            SoilLayer("Aanvulling", 0.5, 17.0, 19.0, 1e-5, 0.6, 0.05, 0.01, 15000, 1e-4),
            SoilLayer("Zand", 2.0, 17.5, 20.0, 1e-4, 0.5, 0.02, 0.005, 30000, 1e-2),
            SoilLayer("Klei", 3.0, 16.0, 18.5, 1e-9, 1.0, 0.30, 0.06, 3000, cv),
            SoilLayer("Zand diep", 4.5, 18.0, 20.5, 5e-4, 0.45, 0.01, 0.003, 40000, 1e-2),
        ]
    )
    t_days = np.linspace(1, 365*5, 500)
    s_t = compute_settlement_vs_time(prof_temp, drawdown=1.5, times_days=t_days)
    s_ult = compute_total_settlement(prof_temp, 1.5)[0]
    idx_50 = np.argmax(s_t >= 0.5 * s_ult)
    t_50_days.append(t_days[idx_50])

plt.figure(figsize=(8, 4))
plt.semilogx(cv_values, t_50_days, '^--', color='darkred')
plt.xlabel("Clay Consolidation Coefficient Cv [m²/s]")
plt.ylabel("Time to 50% Consolidation [days]")
plt.title("Sensitivity 4: Consolidation Time vs Cv")
plt.grid(True, linestyle=":")
plt.show()

In [ ]:
# Sensitivity 5: Well Count
well_counts = [2, 4, 6, 8, 12]
drawdowns_wells = []
for n_w in well_counts:
    w_list = []
    q_per_well = 0.003 / n_w
    for i in range(n_w):
        angle = 2 * np.pi * i / n_w
        wx = 6.0 * np.cos(angle)
        wy = 5.0 * np.sin(angle)
        w_list.append(Well(x=wx, y=wy, Q=q_per_well))
    cfg_temp = DewateringConfig(wells=w_list, target_drawdown_mtaw=1.5, original_gwl_mtaw=4.0, pumping_duration_days=90)
    d_temp = compute_drawdown_at_points([b_pts[0]], cfg_temp, profile)[0]
    drawdowns_wells.append(d_temp)

plt.figure(figsize=(8, 4))
plt.bar([str(n) for n in well_counts], drawdowns_wells, color='navy', alpha=0.7)
plt.xlabel("Number of Pumping Wells")
plt.ylabel("Drawdown at Building Center [m]")
plt.title("Sensitivity 5: Drawdown Uniformity vs Well Count")
plt.grid(True, linestyle=":", axis='y')
plt.show()

## §8 Verification & Sanity Checks (Verificatie)

In [ ]:
# Sanity check assertions
assert 0.0 < drawdowns_b[0] <= dewatering.target_drawdown, "Drawdown out of physical range"
assert 0.001 <= total_s_m <= 0.100, f"Settlement {total_s_m*1000:.1f} mm outside residential range 1-100mm"
assert assessment.damage_category in range(6), "Invalid damage category"
assert np.all(S_grid >= 0), "Drawdown grid contains negative values"
print("All worked example verification checks PASSED successfully!")